# Ways to Solve a System of Linear Equations.
In this notebook, I tried to to implement methods to solve a System of Linear Equations:
$$ Ax = b $$
- Richardson Method
- Jacobi Method
- Gauss-Seidel Method
- Successive Over-Relaxation (SOR)
- Krylov Subspace Methods (modern)

## Init

In [1]:
import numpy as np

## Methods

### 1. Richardson Method

**Core idea**
1. Move in the direction of the residual:
$$ r_{k} = b - Ax_{k} $$
2. Then update:
$$ x_{k+1} = x_{k} + \omega(b - Ax_{k}) $$

**Intuition**
- Residual := How wrong you are.
- Correct the solution in that direction.

In [2]:
def richardson(A, b, x0, omega, tol=1e-8, max_iter=1000):
    x = x0.copy()
    for _ in range(max_iter):
        r = b - A @ x   # matrix multiplication, can use np.matmul(A, x) too
        x_new = x + omega * r
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

### 2. Jacobi Method

**Idea**
1. Split matrix: L/U decomposition
$$ A = D + (L + U) $$
   - D: diagonal mx
   - L, U: L/U parts
2. Update:
$$ x_{i}^{k+1} = \frac{1}{A_{ii}}(b_{i} - \sum_{j \ne i}A_{ij}x_{j}^{(k)}) $$

**Intuition**
- Each variable is updated using only old values.
- Fully parallelisable.

In [3]:
def jacobi(A, b, x0, tol=1e-8, max_iter=1000):
    D = np.diag(A)
    R = A - np.diagflat(D)

    x = x0.copy()
    for _ in range(max_iter):
        x_new = (b - R @ x) / D
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

### 3. Gauss-Seidel Method

**Idea**
- Same as Jacobi but:
Use updated values immediately
$$ x_{i}^{k+1} = \frac{1}{A_{ii}}(b_{i} - \sum{j<i}A_{ij}x_{j}^{(k+1)} - \sum{j>i}A_{ij}x_{j}^{(k)}) $$

**Intuition**
- Jacobi = Wait for the next round
- Gauss-Seidel = Use fresh into instantly

In [4]:
def gauss_seidel(A, b, x0, tol=1e-8, max_iter=1000):
    x = x0.copy()
    n = len(b)

    for _ in range(max_iter):
        x_new = x.copy()
        for i in range(n):
            s1 = np.dot(A[i, :i], x_new[:i])
            s2 = np.dot(A[i, i+1:], x[i+1:])
            x_new[i] = (b[i] - s1 - s2) / A[i, i]
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

### 4. Successive Over-Relaxation (SOR)

**Idea**
- Accelerate Gauss-Seidel with a factor $\omega$
$$ x_{i}^{(k+1)} = (1 - \omega)x_{i}^{(k)} + \frac{\omega}{A_{ii}}(b_{i} - \sum{j<i}A_{ij}x_{j}^{(k+1)} - \sum{j>i}A_{ij}x_{j}^{(k)}) $$

**Intuition**
- $\omega = 1$ -> Gauss-Seidel
- $\omega > 1$ -> accelrate
- $\omega < 1$ -> stabilise

In [5]:
def sor(A, b, x0, omega, tol=1e-8, max_iter=1000):
    x = x0.copy()
    n = len(b)

    for _ in range(max_iter):
        x_new = x.copy()
        for i in range(n):
            s1 = np.dot(A[i, :i], x_new[:i])
            s2 = np.dot(A[i, i+1:], x[i+1:])
            x_new[i] = (1 - omega)*x[i] + (omega / A[i, i]) * (b[i] - s1 - s2)
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

### 5. Krylov Subspace Methods

**Idea**
- Instead of updating coordinate directly, search a smart subspace:
$$ \mathcal{K}_{k}(A, r_{0}) = span \{r_{0}, Ar_{0}, A^{2}r_{0},...\} $$

**Intuition**
- Build a space of directions:
  - residual
  - residual transformed by $A$
  - transformed again...

In [6]:
def conjugate_gradient(A, b, x0, tol=1e-8, max_iter=1000):
    x = x0.copy()
    r = b - A @ x
    p = r.copy()

    for _ in range(max_iter):
        Ap = A @ p
        alpha = (r @ r) / (p @ Ap)

        x_new = x + alpha * p
        r_new = r - alpha * Ap

        if np.linalg.norm(r_new) < tol:
            return x_new
        
        beta = (r_new @ r_new) / (r @ r)
        p = r_new + beta * p
        x, r = x_new, r_new

    return x

## Comparison Summary

| Method | Speed | Parallel | Stability | Use Case |
|--------|-------|----------|-----------|----------|
| Richardson | ❌ slow | ✅ | ❌ | Study |
| Jacobi | ❌ slow | ✅ | ❌ | Simple systems |
| Gauss-Seidel | ❌ medium | ❌ | ✅ | Small systems |
| SOR | ✅ fast | ❌ | ❌ | PDEs |
| Krylov Methods | 🚀 fastest | ❌ | ✅ | Large systems |